[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/notebooks/NB06_msa_generation_qc.ipynb)

> **Opening this notebook from Canvas:** Select **File → Save a copy in Drive** before you begin. Work in your saved copy—not in the repository preview.

**Notebook ID:** `NB06_msa_generation_qc`  
**Course release:** Fall 2026  
**Version:** 1.0  
**Updated:** 2026-09-04


# NB06 — Pairwise distances, guide trees, and multiple-sequence alignment

**Biological question:** How does a progressive MSA use pairwise evidence, and how can we judge whether its columns are trustworthy?

## Learning goals

By the end, you should be able to:

1. Calculate why `N` sequences require `N(N−1)/2` unique pairwise comparisons.
2. Convert pairwise identity to the introductory distance `D = 1 − I`.
3. Explain how a guide tree determines progressive alignment order.
4. Distinguish a guide tree from a phylogenetic tree.
5. Identify conserved, variable, and gap-rich MSA columns.

**Inputs:** an unaligned seven-sequence teaching set and its instructor-curated reference alignment  
**Outputs:** pairwise identity/distance tables, guide tree, and MSA column-QC table

> The sequences are synthetic cytochrome-c-inspired teaching homologs. They are designed to reveal the algorithm's logic; they are not labeled as species observations.


## 1. Add the tools and connect Google Drive

Python is the language; Biopython, NumPy, pandas, and Matplotlib are toolkits. You do not need to memorize the code today. Read each cell's question, run it, and inspect the result.


In [ ]:
%pip install -q biopython

from pathlib import Path
from urllib.request import urlretrieve
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from Bio import Align, AlignIO, Phylo, SeqIO
from Bio.Align import substitution_matrices
from Bio.Phylo.TreeConstruction import DistanceMatrix, DistanceTreeConstructor

from google.colab import drive
drive.mount("/content/drive")
print("Tools and Google Drive are ready.")


## 2. Locate the course folders

The notebook recognizes the standard student folder and the instructor's `Teaching` folder. It creates only missing folders and never overwrites an existing data file.


In [ ]:
COURSE_FOLDER_NAME = "BIOINFO4-5203-F26"
COURSE_RELEASE = "Fall 2026"
REPOSITORY_BRANCH = "main"
NOTEBOOK_ID = "NB06_msa_generation_qc"
NOTEBOOK_VERSION = "1.0"

candidate_course_dirs = [
    Path("/content/drive/MyDrive") / COURSE_FOLDER_NAME,
    Path("/content/drive/MyDrive/Teaching") / COURSE_FOLDER_NAME,
]
existing_course_dirs = [path for path in candidate_course_dirs if path.exists()]
COURSE_DIR = existing_course_dirs[0] if existing_course_dirs else candidate_course_dirs[0]

DATA_DIR = COURSE_DIR / "Data" / NOTEBOOK_ID
OUTPUT_DIR = COURSE_DIR / "Outputs" / NOTEBOOK_ID
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UNALIGNED_PATH = DATA_DIR / "cytochrome_c_teaching_unaligned.fasta"
ALIGNED_PATH = DATA_DIR / "cytochrome_c_teaching_reference_alignment.fasta"

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/RobBurnap/"
    "Bioinformatics-MICR4203-MICR5203/"
    f"{REPOSITORY_BRANCH}/data/{NOTEBOOK_ID}/"
)

print("Course folder :", COURSE_DIR)
print("Data folder   :", DATA_DIR)
print("Output folder :", OUTPUT_DIR)


## 3. Obtain the starter data

If a starter file is missing, the public course copy is downloaded into this notebook's `Data` folder. Existing files are left untouched.


In [ ]:
if not UNALIGNED_PATH.exists():
    urlretrieve(DATA_BASE_URL + "cytochrome_c_teaching_unaligned.fasta", UNALIGNED_PATH)
    print("Downloaded:", UNALIGNED_PATH.name)
else:
    print("Using existing:", UNALIGNED_PATH.name)
if not ALIGNED_PATH.exists():
    urlretrieve(DATA_BASE_URL + "cytochrome_c_teaching_reference_alignment.fasta", ALIGNED_PATH)
    print("Downloaded:", ALIGNED_PATH.name)
else:
    print("Using existing:", ALIGNED_PATH.name)

## 4. Validate the homolog panel

Seven sequences require `7 × 6 / 2 = 21` unique pairwise comparisons. The formula avoids self-comparisons and double-counting A–B plus B–A.


In [ ]:
records = list(SeqIO.parse(UNALIGNED_PATH, "fasta"))
names = [record.id for record in records]
sequences = [str(record.seq) for record in records]

N = len(records)
expected_pairs = N * (N - 1) // 2
assert N == 7
assert len(set(names)) == N

print(f"Sequences: {N}")
print(f"Unique pairwise alignments: {expected_pairs}")
for record in records:
    print(f"{record.id:18s} {len(record.seq):3d} aa")


## 5. Perform every global pairwise alignment

Like the Clustal-family workflow introduced in lecture, we first compare every pair globally. BLOSUM62 rewards chemically plausible protein substitutions; affine gap penalties make opening a gap more costly than extending one.


In [ ]:
pair_aligner = Align.PairwiseAligner()
pair_aligner.mode = "global"
pair_aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
pair_aligner.open_gap_score = -10
pair_aligner.extend_gap_score = -0.5

def identity_from_alignment(alignment):
    paired = identical = 0
    for i, j in zip(*alignment.indices):
        if i >= 0 and j >= 0:
            paired += 1
            identical += alignment.target[i] == alignment.query[j]
    return identical / paired

identity = pd.DataFrame(np.eye(N), index=names, columns=names)
pair_rows = []

for i in range(N):
    for j in range(i + 1, N):
        alignment = pair_aligner.align(sequences[i], sequences[j])[0]
        value = identity_from_alignment(alignment)
        identity.iloc[i, j] = identity.iloc[j, i] = value
        pair_rows.append({"sequence_1": names[i], "sequence_2": names[j], "identity": value, "distance": 1 - value})

pairwise_results = pd.DataFrame(pair_rows)
assert len(pairwise_results) == expected_pairs
display(pairwise_results.sort_values("distance").head(8).round(3))


## 6. Turn identity into an introductory distance

For this activity, `D = 1 − I`. Identical sequences have distance 0; lower identity produces a larger distance. This simple distance is intuitive but does not correct for multiple substitutions at the same site.


In [ ]:
distance = 1 - identity
np.fill_diagonal(distance.values, 0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, table, title, cmap in [
    (axes[0], identity, "Pairwise identity", "Blues"),
    (axes[1], distance, "D = 1 − identity", "Oranges"),
]:
    image = ax.imshow(table, vmin=0, vmax=1, cmap=cmap)
    ax.set_xticks(range(N), names, rotation=75, ha="right")
    ax.set_yticks(range(N), names)
    ax.set_title(title)
    fig.colorbar(image, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pairwise_identity_distance.png", dpi=160)
plt.show()


## 7. Build a guide tree

The guide tree is a **work plan**: the most similar sequences or profiles are aligned first, then progressively combined. It is not automatically a phylogenetic tree because its distances and alignment procedure were chosen to guide an algorithm, not to test an evolutionary model.


In [ ]:
lower_triangle = []
for i in range(N):
    lower_triangle.append([float(distance.iloc[i, j]) for j in range(i + 1)])

bio_distance = DistanceMatrix(names=names, matrix=lower_triangle)
guide_tree = DistanceTreeConstructor().nj(bio_distance)

Phylo.draw_ascii(guide_tree)
Phylo.write(guide_tree, OUTPUT_DIR / "guide_tree.nwk", "newick")

closest = pairwise_results.sort_values("distance").iloc[0]
print("First plausible pair to combine:", closest.sequence_1, "+", closest.sequence_2)
print("Distance:", round(closest.distance, 3))


### “Once a gap, always a gap”

In a classical progressive alignment, early alignment decisions are usually not revisited. A mistaken early gap can therefore be propagated through later profile alignments. This is why sequence order, scoring, and homolog quality matter.

**Prediction:** Which sequence looks most likely to create an unstable early decision, and why?


## 8. Inspect a curated reference MSA

To keep this first MSA activity reproducible, we inspect an instructor-provided aligned result rather than depending on an external web server. A later notebook can replace it with the class's own homolog set and alignment program.


In [ ]:
msa = AlignIO.read(ALIGNED_PATH, "fasta")
assert len(msa) == N
assert len({len(record.seq) for record in msa}) == 1

print("Sequences:", len(msa))
print("Alignment columns:", msa.get_alignment_length())
print(msa)


## 9. Score each alignment column

- **gap fraction:** fraction of sequences containing `-`
- **conservation:** frequency of the most common non-gap residue
- **consensus:** most common non-gap residue

High conservation can support functional or structural importance. It does not prove a mechanism by itself.


In [ ]:
column_rows = []
for column_index in range(msa.get_alignment_length()):
    column = [str(record.seq[column_index]) for record in msa]
    residues = [aa for aa in column if aa != "-"]
    counts_in_column = pd.Series(residues).value_counts()
    consensus = counts_in_column.index[0] if len(counts_in_column) else "-"
    conservation = counts_in_column.iloc[0] / len(residues) if residues else np.nan
    column_rows.append({
        "column_1_based": column_index + 1,
        "consensus": consensus,
        "conservation": conservation,
        "gap_fraction": column.count("-") / N,
    })

msa_qc = pd.DataFrame(column_rows)
display(msa_qc.sort_values(["conservation", "gap_fraction"], ascending=[False, True]).head(12))


## 10. Find conserved motifs and uncertain columns

The synthetic panel retains a cytochrome-c-like `CXXCH` motif. Find its consensus position, then inspect any gap-rich or weakly conserved columns nearby.


In [ ]:
consensus = "".join(msa_qc.consensus)
motif_start = consensus.find("C")
print("Consensus:", consensus)
print("Most conserved columns:")
display(msa_qc.query("conservation == 1.0 and gap_fraction == 0").head(15))
print("Columns containing gaps or conservation below 0.75:")
display(msa_qc.query("gap_fraction > 0 or conservation < 0.75"))


## 11. Decide whether the MSA is fit for purpose

Answer in complete sentences:

1. Which pair was most similar, and where does it join the guide tree?
2. Which sequence behaves as the outlier?
3. Which columns would you trust most for inferring conserved function or structure?
4. Which columns deserve caution, and why?
5. Why must the guide tree not be presented as a final evolutionary tree?

**Your answers:**


In [ ]:
pairwise_results.to_csv(OUTPUT_DIR / "pairwise_comparisons.tsv", sep="	", index=False)
identity.to_csv(OUTPUT_DIR / "pairwise_identity_matrix.tsv", sep="	")
distance.to_csv(OUTPUT_DIR / "pairwise_distance_matrix.tsv", sep="	")
msa_qc.to_csv(OUTPUT_DIR / "msa_column_qc.tsv", sep="	", index=False)

pd.DataFrame([
    ["notebook_id", NOTEBOOK_ID],
    ["notebook_version", NOTEBOOK_VERSION],
    ["sequence_count", N],
    ["unique_pairwise_comparisons", expected_pairs],
    ["substitution_matrix", "BLOSUM62"],
    ["distance_definition", "1 - pairwise identity"],
    ["msa_status", "synthetic instructor-curated teaching alignment"],
], columns=["parameter", "value"]).to_csv(OUTPUT_DIR / "run_parameters.tsv", sep="	", index=False)

print("Files created:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)


## Exit ticket

Complete both statements:

> A guide tree is useful because __________, but it is not necessarily a phylogenetic tree because __________.

> I would distrust an MSA column when __________ because __________.
